In [1]:
import time
import numpy as np
import pandas as pd
import optuna
import mlflow
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os, json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Load the final files

In [2]:
train = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/train_final.csv')
val   = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/val_final.csv')
test  = pd.read_csv('C:/Projects/Housing-Price-Prediction/data/processed/test_final.csv')

In [3]:
non_features = ['zipcode', 'price', 'year', 'city', 'date']

X_train, y_train = train.drop(columns=non_features), train['price']
X_val,   y_val   = val.drop(columns=non_features),   val['price']
X_test,  y_test  = test.drop(columns=non_features),  test['price']

print('zipcode' in X_train.columns, 'year' in X_train.columns)   # both should be False
print('median_sale_price' in X_train.columns)                    # should be True
print(list(X_train.columns) == list(X_val.columns) == list(X_test.columns))  # should be True

False False
True
True


In [5]:
def score(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs(y_true - y_pred) / y_true) * 100
    r2   = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE_%': mape, 'R2': r2}

In [6]:
t0 = time.time()
base = CatBoostRegressor(random_seed=42, verbose=0, allow_writing_files=False)
base.fit(X_train, y_train)
print("fit time (minutes):", round((time.time() - t0) / 60, 1))

print({k: round(v, 2) for k, v in score(y_val, base.predict(X_val)).items()})

fit time (minutes): 1.0
{'RMSE': np.float64(63662.47), 'MAE': 30003.7, 'MAPE_%': np.float64(6.9), 'R2': 0.97}


In [8]:
os.makedirs('C:/Projects/Housing-Price-Prediction/tracking', exist_ok=True)
mlflow.set_tracking_uri("sqlite:///C:/Projects/Housing-Price-Prediction/tracking/mlflow.db")
mlflow.set_experiment("catboost_tuning_final_features")
optuna.logging.set_verbosity(optuna.logging.WARNING)

2026/11/27 11:35:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/27 11:35:31 INFO mlflow.store.db.utils: Updating database tables
2026/11/27 11:35:34 INFO mlflow.tracking.fluent: Experiment with name 'catboost_tuning_final_features' does not exist. Creating a new experiment.


In [ ]:
def objective(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 2000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.02, 0.3, log=True),
        'depth':               trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1, 20),
        'random_strength':     trial.suggest_float('random_strength', 0.5, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'border_count':        trial.suggest_int('border_count', 64, 254),
    }

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        t0 = time.time()
        model = CatBoostRegressor(**params, random_seed=42, verbose=0,
                                  allow_writing_files=False)
        model.fit(X_train, y_train)
        s = score(y_val, model.predict(X_val))
        minutes = (time.time() - t0) / 60

        mlflow.log_params(params)
        mlflow.log_metrics({'rmse': float(s['RMSE']), 'mae': float(s['MAE']),
                            'mape': float(s['MAPE_%']), 'r2': float(s['R2']),
                            'fit_minutes': minutes})

        trial.set_user_attr('mae', float(s['MAE']))
        trial.set_user_attr('mape', float(s['MAPE_%']))

    return float(s['RMSE'])

In [ ]:
study = optuna.create_study(
    study_name='catboost_final_features',
    storage='sqlite:///C:/Projects/Housing-Price-Prediction/tracking/optuna.db',
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

In [ ]:
best = study.best_trial
print("best RMSE:", round(best.value), "| MAE:", round(best.user_attrs['mae']),
      "| MAPE:", round(best.user_attrs['mape'], 2))
print("best params:", best.params)
print("\nbaseline to beat: RMSE 63,785 | MAE 30,269 | MAPE 6.97")

trials = study.trials_dataframe()
trials['mae'] = [t.user_attrs.get('mae') for t in study.trials]
print(trials.sort_values('value').head(10)[['number', 'value', 'mae', 'params_depth',
      'params_learning_rate', 'params_iterations']].round(3).to_string(index=False))

In [ ]:
os.makedirs('C:/Projects/Housing-Price-Prediction/data/results', exist_ok=True)

In [ ]:
with open('C:/Projects/Housing-Price-Prediction/data/results/best_params.json', 'w') as f:
    json.dump(study.best_trial.params, f, indent=2)

trials = study.trials_dataframe()
trials['mae'] = [t.user_attrs.get('mae') for t in study.trials]
trials.to_csv('C:/Projects/Housing-Price-Prediction/data/results/optuna_trials.csv', index=False)